## Vorteile von Case Classes
- Case Classes sind *der* Shortcut, wenn man bequem Klassen bauen will
- automatisch generierte Methoden wie `toString()`, `equals()`, `hashCode()`, usw. ...
- Automatisches **Companion Object** mit:
  - `apply`: Objekterzeugung ohne `new`
  - `unapply`: für Pattern Matching
  
## `apply` und `unapply`
### `apply`
- ermöglicht Objekterzeugung ohne `new`
- Interner Aufruf von `apply`, z. B.:
```scala
case class Person(name: String)
val bercan = Person("bercan") // ruft intern: Person.apply("bercan")
```
### .unapply()
- wird im *Companion Object* definiert, um Objekte in ihre Bestandteile zu zerlegen: Rückgabetyp ist dabei ebenfalls `Option[T]` (siehe nächstes Tutorium)
- wird implizit bei Pattern Matching aufgerufen:



In [ ]:
case class Animal(name:String)

class Plant(name:String)

var elvis :Animal = new Animal("Elvis")
var elvis2 :Animal = new Animal("Elvis2")



var printNameAndotherStuffAnimal: Animal => Unit = (animal:Animal) => animal match {
    case Animal("Elvis")              => println(s"Elvis ist ein guter Junge") 
    case Animal(name)   => println(s"$name ist bestimmt auch toll ")
    //case Plant(name)    => println(s"$name ist eine Pflanze der Gattung  ")  // funktioniert nicht weil die unapply Methode fehlt ohne case class
}
printNameAndotherStuffAnimal(elvis2)




Elvis2 ist bestimmt auch toll 


defined class Animal
elvis: Animal = Animal(name = "Elvis")
elvis2: Animal = Animal(name = "Elvis2")
printNameAndotherStuffAnimal: Animal => Unit = ammonite.$sess.cmd1$Helper$$Lambda/0x000002b3818a2000@53ec593

In [ ]:
class Circle(val radius: Double)
class Rectangle(val width: Double, val height: Double)

object Circle {
    def unapply(c: Circle): Option[Double] = Some(c.radius)
}

object Rectangle {
    def unapply(c: Rectangle): Option[(Double, Double)] = Some(c.width,c.height)
}

def calculateArea(shape: Any): Option[Double] = shape match {
  case Circle(radius)              => Some(3.14 * java.lang.Math.pow(radius, 2))
  //case Rectangle(width, height)    => Some(width * height) // funktioniert nicht weil die unapply Methode fehlt
    case _ => None
}


defined class Circle
defined class Rectangle
defined object Circle
defined object Rectangle
defined function calculateArea
calculateArea: Any => Option[Double] = ammonite.$sess.cmd2$Helper$$Lambda/0x000002b3818a31f8@581200df

### Eta-Expansion
- wandelt eine Methode in ein Funktionsobjekt um => so können Methoden wie *First-Class Citizens* verwendet werden
- Umwandlung erfolgt über den Unterstrich-Operator: `_` (Scala erzeugt dann ein **Funktionsobjekt**, das intern den Methodenaufruf speichert)

In [ ]:
def add(x: Int, y: Int): Int = x + y

//val addFunction: (Int,Int) => Int = (x,y) => x+y //so würde die add-Methode als Funktion aussehen 

val addFunction = add _ // Scala kann das explizit mit dem _ Operator hinter dem Methodennamen

println(addFunction(2, 3))           // funktioniert
println(addFunction.apply(2, 3))     // funktioniert auch, da addFunction ein Funktionsobjekt mit .apply ist

ammonite.$sess.cmd33$Helper$$Lambda/0x00000278a29c8c60@6f64f217
ammonite.$sess.cmd33$Helper$$Lambda/0x00000278a29c86a0@1ef37921
5
5


defined function add
addFunction: (Int, Int) => Int = ammonite.$sess.cmd33$Helper$$Lambda/0x00000278a29c86a0@1ef37921

### Partiell angewandte Funktionen
- nehmen eine Funktion und fixieren Argumenten in ihr => so erzeugen sie eine neue Funktion, die noch auf die restlichen Parameter (*Plural*) wartet

In [4]:
def multiply = (x:Int, y:Int, z:Int) => x*y*z
val multiplyBy2And3 = multiply(2, _, 3:Int) //2 und 3 wurden fixiert.
println(multiplyBy2And3(3)) //multiplyBy2And3 ist jetzt eine Funktion mit nur einem InputParameter


18


defined function multiply
multiplyBy2And3: Int => Int = ammonite.$sess.cmd4$Helper$$Lambda/0x000002b38195d410@1f20924c

### Currying
- nimmt eine Funktion und zerlegt sie in eine Kette von Funktionen, die jeweils nur einen Parameter (*Singular*) gleichzeitig akzeptiert
- Die Methoden die hierbei erzeugt werden können auch alle durch partiell angewandte Methoden erzeugt werden.
- *Tipp von mir: wiederholt zuerst *Funktion Höherer Ordnung*, wenn ihr euch schwer mit Currying tut*

In [ ]:
val add: (Int, Double)=> Double = (x,y)=> x+y //dieses add nimmt zwei Werte gleichzeitig entgegen
val addCurried:Int => (Double => Double) = (x) => ((y)=> x+y) //Diese Funktion bekommt einen Wert und returned eine Funktion die wieder einen Wert bekommt 

val add5ToSomeValue:(Double => Double) = addCurried(5) //addcurried returned als Funktion vom Typ (Double=>Double)
println(add5ToSomeValue(4.0))

val Mult3Params: (Int,Double,Int) => Double = (x,y,z) => x*y*z  //das gleiche geht auch mit mehr Parametern
val mult3ParamCurriedOurVersion:Int => (Double => (Int  => Double) )= (x) => ((y) => ((z)=> x*y*z))
val mult3ParamCurried = Mult3Params.curried //Jedes Funktionsobjekt hat auch die Methode curried welche das automatisch generieren kann
mult3ParamCurried

val newFunc:Int  => Double = (mult3ParamCurried(5)(5.0))
newFunc(5)


9.0
11.0
8.0
15.0


add: (Int, Double) => Double = ammonite.$sess.cmd42$Helper$$Lambda/0x00000278a29ef9a8@2aa43cc0
addCurried: Int => Double => Double = ammonite.$sess.cmd42$Helper$$Lambda/0x00000278a29f4000@7a3ef89e
add5ToSomeValue: Double => Double = ammonite.$sess.cmd42$$Lambda/0x00000278a29f43b8@1fe79684
Mult3Params: (Int, Double, Int) => Double = ammonite.$sess.cmd42$Helper$$Lambda/0x00000278a29f4790@253abbed
mult3ParamCurried: Int => Double => Int => Double = scala.Function3$$Lambda/0x00000278a29d7730@70f67368
res42_6: Int => Double => Int => Double = scala.Function3$$Lambda/0x00000278a29d7730@70f67368
mult3ParamCurriedOurVersion: Int => Double => Int => Double = ammonite.$sess.cmd42$Helper$$Lambda/0x00000278a29f49c8@10085c96
newFunc: Int => Double = scala.Function3$$Lambda/0x00000278a29e4000@60ffd757
res42_9: Double = 125.0
add5Plus: Double => Double = ammonite.$sess.cmd42$$Lambda/0x00000278a29f43b8@3b12dc82